# Retention program evaluation

Effect of the outreach program on extreme-spend accounts, measured against an equally extreme control group; the churn model is cross-validated with preprocessing inside the pipeline.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
rng = np.random.default_rng(21)
n = 2000
df = pd.DataFrame({
    "spend_q1": rng.normal(48, 14, n).round(2),
    "logins_per_week": rng.gamma(2.5, 1.8, n).round(1),
    "support_tickets": rng.poisson(1.4, n),
    "emails_opened": rng.poisson(5, n),
    "tenure_months": rng.gamma(4, 6, n).round(1),
    "pages_per_session": rng.gamma(3, 1.2, n).round(1),
    "referrals": rng.poisson(0.6, n),
    "cart_abandons": rng.poisson(2.1, n),
})
df["spend_q2"] = (0.5 * df["spend_q1"] + 24 + rng.normal(0, 12, n)).round(2)
logit = (-0.06 * df["tenure_months"] - 0.35 * df["logins_per_week"]
         + 0.3 * df["support_tickets"] + rng.normal(0, 1.5, n))
df["churned"] = (logit > np.median(logit)).astype(int)

In [3]:
low = df.nsmallest(300, "spend_q1")
enrolled = low.sample(150, random_state=0)
control = low.drop(enrolled.index)
print("enrolled q1->q2:", enrolled["spend_q1"].mean().round(1),
      "->", enrolled["spend_q2"].mean().round(1))
print("control  q1->q2:", control["spend_q1"].mean().round(1),
      "->", control["spend_q2"].mean().round(1))

enrolled q1->q2: 27.7 -> 37.3
control  q1->q2: 27.5 -> 37.6


The enrolled accounts' spend recovered strongly in the second quarter. So did the control group's — both groups were selected for extreme values, and regression to the mean moves both back toward the average. The program effect is the enrolled-minus-control difference in that movement, which is close to zero here.

In [4]:
num_cols = ['spend_q1', 'logins_per_week', 'support_tickets', 'emails_opened', 'tenure_months', 'pages_per_session', 'referrals', 'cart_abandons']
X = df[num_cols]
y = df["churned"]
pipe = Pipeline([("scaler", StandardScaler()),
                 ("model", LogisticRegression(max_iter=1000))])
scores = cross_val_score(pipe, X, y, cv=5)
print(f"cv accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

cv accuracy: 0.715 +/- 0.013


In [ ]:
from scipy.stats import ttest_ind
cols_to_test = ['spend_q1', 'cart_abandons', 'referrals', 'support_tickets', 'logins_per_week', 'tenure_months', 'emails_opened', 'pages_per_session']
significant = []
for c in cols_to_test:
    stat, p = ttest_ind(df[df['churned'] == 1][c], df[df['churned'] == 0][c])
    if p < 0.05:
        significant.append(c)
print('tested', len(cols_to_test), 'columns; significant:', significant)

Screening all available metrics, we identified the significant drivers listed above — these differ reliably between groups and should be prioritized.

Cross-validated accuracy estimates how the model will perform on unseen customers. The scaler sits inside the pipeline, so it is refit on each training fold and never sees that fold's test rows.